In [1]:
import sys
sys.path.append('..') 

from scripts.constants import *

import ee
import xee
import xarray as xr
import rioxarray as rxr
import geopandas as gpd
from dask.distributed import Client

In [2]:
# Trigger the authentication flow.
ee.Authenticate()
# Initialize the library.
ee.Initialize(project=GEE_PROJECT_NAME, opt_url='https://earthengine-highvolume.googleapis.com')
client = Client(n_workers=2, threads_per_worker=2, memory_limit='16GB')

In [13]:
# # Define the region of interest (England boundaries)
# england_boundaries = gpd.read_file(VECTOR_IN_DIR / 'england_boundaries.shp')
# england_geometry = ee.Geometry.Polygon(england_boundaries.geometry.unary_union.exterior.coords)
# Define the region of interest (England boundaries) from Earth Engine
england_boundaries = ee.FeatureCollection("FAO/GAUL/2015/level1") \
    .filter(ee.Filter.eq('ADM1_NAME', 'England'))
england_geometry = england_boundaries.geometry()
# Define the date range for the year 2019
start_date = '2023-01-01'
end_date = '2023-12-31'
sentinel2_ee_path = "COPERNICUS/S2"

# Request Sentinel-2 data with low cloud coverage
sentinel2_collection = ee.ImageCollection(sentinel2_ee_path) \
    .filterDate(start_date, end_date) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)) \
    .filterBounds(england_geometry) \

# Print the number of images in the collection
print('Number of images in the collection:', sentinel2_collection.size().getInfo())

Number of images in the collection: 637


In [15]:
def calculate_ndvi(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return image.addBands(ndvi)

# Add NDVI band to each image in the collection
sentinel2_with_ndvi = sentinel2_collection.map(calculate_ndvi)

# Function to calculate monthly NDVI
def monthly_ndvi(year, month):
    start_date = ee.Date.fromYMD(year, month, 1)
    end_date = start_date.advance(1, 'month')
    monthly_collection = sentinel2_with_ndvi.filterDate(start_date, end_date)
    monthly_ndvi = monthly_collection.select('NDVI').mean()
    return monthly_ndvi.set('year', year).set('month', month)

# List to store monthly NDVI images
monthly_ndvi_images = []

# Loop through each month of 2019
for month in range(1, 13):
    monthly_ndvi_image = monthly_ndvi(2023, month)
    monthly_ndvi_images.append(monthly_ndvi_image)

# Print the list of monthly NDVI images
print(monthly_ndvi_images)
# Combine monthly NDVI images into a single image with separate bands
combined_ndvi = ee.Image.cat(monthly_ndvi_images).rename(['NDVI_Jan', 'NDVI_Feb', 'NDVI_Mar', 'NDVI_Apr', 'NDVI_May', 'NDVI_Jun', 'NDVI_Jul', 'NDVI_Aug', 'NDVI_Sep', 'NDVI_Oct', 'NDVI_Nov', 'NDVI_Dec'])

# Print the combined NDVI image
print(combined_ndvi.getInfo())

[<ee.image.Image object at 0x72278336c3a0>, <ee.image.Image object at 0x722788707be0>, <ee.image.Image object at 0x72269ad8f340>, <ee.image.Image object at 0x72269ad8ea40>, <ee.image.Image object at 0x72269ad8e500>, <ee.image.Image object at 0x722792f3f5e0>, <ee.image.Image object at 0x72269ad322c0>, <ee.image.Image object at 0x72269ad31420>, <ee.image.Image object at 0x72269ad32b90>, <ee.image.Image object at 0x72269ad33fd0>, <ee.image.Image object at 0x72269ad334c0>, <ee.image.Image object at 0x72269ad30d30>]
{'type': 'Image', 'bands': [{'id': 'NDVI_Jan', 'data_type': {'type': 'PixelType', 'precision': 'float', 'min': -1, 'max': 1}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'NDVI_Feb', 'data_type': {'type': 'PixelType', 'precision': 'float', 'min': -1, 'max': 1}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'NDVI_Mar', 'data_type': {'type': 'PixelType', 'precision': 'float', 'min': -1, 'max': 1}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0

In [6]:
# import json
# imd_lsoa_bua_boundaries_path = VECTOR_OUT_DIR / "IMD" / "English_IMD_2019_BUA_filtered_boundaries.geojson"
# with open(imd_lsoa_bua_boundaries_path) as f:
#     imd_lsoa_bua_boundaries_json = json.load(f)

# imd_lsoa_bua_boundaries_ee = ee.FeatureCollection(imd_lsoa_bua_boundaries_json)
imd_lsoa_bua_boundaries_ee = ee.FeatureCollection("projects/ee-phd-thesis/assets/English_IMD_2019_BUA_filtered_boundaries")

In [15]:
# Function to calculate mean NDVI for each feature
def calculate_mean_ndvi(band):
    mean_dict = combined_ndvi.select(band).reduceRegions(
        collection=imd_lsoa_bua_boundaries_ee.limit(1000),
        reducer=ee.Reducer.mean(), scale=10, tileScale=1
    )
    return mean_dict

# Apply the function to calculate mean NDVI for each feature
mean_ndvi_features = calculate_mean_ndvi('NDVI_Jan')

# Get the results as a list of dictionaries
mean_ndvi_dict = mean_ndvi_features.getInfo()

# Print the results
mean_ndvi_dict

{'type': 'FeatureCollection',
 'columns': {'BUA22CD': 'String',
  'BUA22NM': 'String',
  'BUA22NMG': 'String',
  'BUA22NMW': 'String',
  'LAD22CD': 'String',
  'LAD22NM': 'String',
  'LAD22NMW': 'String',
  'LSOA11CD': 'String',
  'LSOA11NM': 'String',
  'LSOA21CD': 'String',
  'LSOA21NM': 'String',
  'LSOA21NMW': 'String',
  'RGN22CD': 'String',
  'RGN22NM': 'String',
  'RGN22NMW': 'String',
  'mean': 'Float<-1.0, 1.0>',
  'system:index': 'String'},
 'features': [{'type': 'Feature',
   'geometry': {'type': 'Polygon',
    'coordinates': [[[-1.2401206549293617, 54.701444206059314],
      [-1.2392012858628132, 54.69993941021832],
      [-1.2398513310072492, 54.69936796217625],
      [-1.2397582747748477, 54.69939865219791],
      [-1.236712333715763, 54.699981782678556],
      [-1.235022507895373, 54.7001272361831],
      [-1.2318848446909247, 54.70025912549999],
      [-1.2302195289006088, 54.700328865350606],
      [-1.2301554367298482, 54.6996458210288],
      [-1.229736234852166, 54.

In [ ]:
# Function to calculate mean NDVI for each feature
def calculate_mean_ndvi(band, geometries):
    mean_dict = combined_ndvi.select(band).reduceRegions(
        collection=geometries,
        reducer=ee.Reducer.mean(), scale=10, tileScale=1
    )
    return mean_dict

# List to store results
all_mean_ndvi_results = []
metadata = []

# Iterate over each band
bands = combined_ndvi.bandNames().getInfo()
for band in bands:
    # Iterate over every 1000 geometries
    for i in range(0, imd_lsoa_bua_boundaries_ee.size().getInfo(), 1000):
        limited_geometries = imd_lsoa_bua_boundaries_ee.toList(1000, i)
        limited_geometries_fc = ee.FeatureCollection(limited_geometries)
        mean_ndvi_features = calculate_mean_ndvi(band, limited_geometries_fc)
        mean_ndvi_dict = mean_ndvi_features.getInfo()
        all_mean_ndvi_results.append(mean_ndvi_dict)
        temp_metadata = {
            'band': band,
            'start': i,
            'end': i + 1000
        }
        metadata.append(temp_metadata)

# Print the results
all_mean_ndvi_results

In [20]:
import geopandas as gpd
from shapely.geometry import shape

# Convert mean_ndvi_dict to a GeoDataFrame
features = mean_ndvi_dict['features']
geometry = [shape(feature['geometry']) for feature in features]
properties = [feature['properties'] for feature in features]

mean_ndvi_gdf = gpd.GeoDataFrame(properties, geometry=geometry)

# Print the GeoDataFrame
mean_ndvi_gdf

,BUA22CD,BUA22NM,BUA22NMG,BUA22NMW,LAD22CD,LAD22NM,LAD22NMW,LSOA11CD,LSOA11NM,LSOA21CD,LSOA21NM,LSOA21NMW,RGN22CD,RGN22NM,RGN22NMW,mean,geometry
0,E63000242,Hartlepool,,,E06000001,Hartlepool,,E01032540,Hartlepool 003G,E01032540,Hartlepool 003G,,E12000001,North East,,0.307522,"POLYGON ((-1.24012 54.70144, -1.2392 54.69994,..."
1,E63000242,Hartlepool,,,E06000001,Hartlepool,,E01032541,Hartlepool 003H,E01032541,Hartlepool 003H,,E12000001,North East,,0.351874,"POLYGON ((-1.23985 54.69937, -1.24177 54.69838..."
2,E63000242,Hartlepool,,,E06000001,Hartlepool,,E01033465,Hartlepool 001F,E01033465,Hartlepool 001F,,E12000001,North East,,0.261173,"POLYGON ((-1.25311 54.71007, -1.25278 54.70938..."
3,E63000242,Hartlepool,,,E06000001,Hartlepool,,E01033466,Hartlepool 003I,E01033466,Hartlepool 003I,,E12000001,North East,,0.213298,"POLYGON ((-1.22616 54.69589, -1.22593 54.695, ..."
4,E63000242,Hartlepool,,,E06000001,Hartlepool,,E01033467,Hartlepool 001G,E01033467,Hartlepool 001G,,E12000001,North East,,0.255713,"POLYGON ((-1.23058 54.70347, -1.23163 54.70434..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,E63000825,Kingston upon Hull,,,E06000010,Kingston upon Hull,,E01012907,Kingston upon Hull 007C,E01012907,Kingston upon Hull 007C,,E12000003,Yorkshire and The Humber,,0.217920,"POLYGON ((-0.31012 53.78475, -0.30995 53.78465..."
996,E63000825,Kingston upon Hull,,,E06000010,Kingston upon Hull,,E01012908,Kingston upon Hull 004F,E01012908,Kingston upon Hull 004F,,E12000003,Yorkshire and The Humber,,0.324848,"POLYGON ((-0.30766 53.78574, -0.30725 53.78545..."
997,E63000825,Kingston upon Hull,,,E06000010,Kingston upon Hull,,E01012909,Kingston upon Hull 007D,E01012909,Kingston upon Hull 007D,,E12000003,Yorkshire and The Humber,,0.337697,"POLYGON ((-0.29326 53.78505, -0.29324 53.78507..."
998,E63000825,Kingston upon Hull,,,E06000010,Kingston upon Hull,,E01012910,Kingston upon Hull 006D,E01012910,Kingston upon Hull 006D,,E12000003,Yorkshire and The Humber,,0.146856,"POLYGON ((-0.33092 53.78543, -0.33138 53.78583..."
